# Ensemble estimators
In previous guides, we have disscused some ML approaches that consists on training many "weak" estimators to build a "strong" estimator, this method is called "ensemble" because combines the predictions of several base estimators built with given learning algorithm in order to improve generalizabilitu over a single estimator. Most popular algorithms are random forests, gradient-boosted trees and early 2000s Adaboost.

In this guide I'm going to show you approaches beyond discussing about single ML techniques like extremly randomized trees. Note that this strategy can be applied to any base learner, from Naive Bayes classifiers to neural nets or SVMs.

## Bagging
Bagging methods is a class of algorithms which build several instancias of base learners on random subsets of the original training set and the aggregate their individual outcomes to form the final output. These methods are used as a way to reduce the variance of a base estimator by introducing randomization into its contruction procedure. Works best for strong and complex models such as fully developed decision trees in contrast to boosting methods whic usually work best with weak models.

### Classification
It uses a majority voting system on individual estimator $h_i(x)$ outcome where the most voted label is the final model output. Having $C$ classes and $n$ estimator, can be expressed as:

$$
    \hat{y}(x) = \operatorname*{arg\,max}_{c \in \{1,\ldots, C \}} \sum_{i=1}^{n} \mathbf{I}[h_i(x) = c]
$$

where the decision function $\mathbf{I}$ is defined as follows:
$$
     \mathbf{I}[h_i(x) = c] = \begin{cases}
 1 & \text{if } h_i(x) = c \\
 0 & \text{otherwise}
\end{cases}
$$

### Regression
The model output is an average of individual outcomes of each estimator.

$$
    \hat{y}(x) = \frac{1}{n} \sum_{i=1}^{n} h_i(x)
$$

### Implementation
In Scikit-learn it can be found at `BaggingClassifier` and `BaggingRegressor` classes to build a meta-estimator from the provided base estimator. The most relevant hyperparameters are:

- **`estimator`:** The base estimator to fit.
- **`n_estimators`:** Number of base estimators in the ensemble.
- **`max_samples`:** Number of samples to draw from the training dataset to each base estimator.
- **`max_features`:** Number of features to draw from training dataset to each base estimator.
- **`boostrap`:** Whether samples are drawn with replacement.
- **`boostrap_features`:** Whether features are drawn with replacement (applies boostrapping to feature set).

In [1]:
from pandas import read_csv
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = BaggingClassifier(estimator=KNeighborsClassifier(n_neighbors=100, metric="euclidean"), n_jobs=-1, n_estimators=5)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

Accuracy: 67.250000 %
Weighted F1-Score: 66.824896 %


## Boosting
I don't want to repeat myself, so I let you the definition given at [gradient boosting trees guide](https://www.kaggle.com/code/criser2013/gradient-boosting-trees)

>Boosting is an ensemble technique that builds a strong model by combining multiple weak learners sequentially.
>
>This is different from bagging, where individual learners are trained independently—often on different samples of the training data—and their predictions are then aggregated.
>
>In boosting, models are trained sequentially. Each new learner is added to improve the predictions made by the ensemble built so far. In other words, instead of training every tree independently, boosting uses the information from previous iterations to determine what the next tree should learn.
>
>The final model can be viewed as a weighted sum of the predictions produced by the individual learners.

In a general formulation boosting models are defined as the following equation:

$$
    F(x) = F_{0}(x) + \sum_{m=1}^{M} h_{m}(x)
$$

where:
- $F_{0}(x)$ is the initial model or base prediction.
- $h_{m}(x)$ is the estimator added at iteration $m$.
- $M$ is the total number of boosting iterations (estimators).

There isn't a general formulation to calculate the output in this kind of models, in classification tasks is common to apply **sigmoid** or **softmax** functions on $F(x)$ to calculate a probability for belonging to a certain class; in contrast, for regression tasks it works different, for instance in gradient based decision trees the final outcome is $F(x)$ but in Ada boost you have to calculate a weighted median over individual estimator's output

### Implementation
Scikit learn doesn't provides a boosting meta-estimator, the only available classes are `AdaBoostClassifier`, `AdaBoostRegressor`, `GradientBoostingClassifier`, `GradientBoostingRegressor`, `HistGradientBoostingClassifier` and `HistGradientBoostingRegressor` for Ada boost and gradient based decision trees. Also `xgboost` package provides an improvement of gradient based decision trees.

## Voting
As the name suggests it mainly designed for classification tasks, however it can be extended fo regression tasks. The idea is the same as bagging estimators combining different models to build a robust model, but in this case we aren't limited to use just the same algorithm for each estimator. We specify each estimator manually and then every one is trained independently from the others without applying any transformation to the training dataset.

### Classification
It has 2 approaches for final output calculation:

1. **Hard voting:** Works such as bagging classifiers using a majority voting system, that chooses the most frequent class at individual estimators output.
2. **Soft voting:** Calculates an average of the class probabilities for each estimator $p_{i}^{(c)}$ and selects the class with the highest probability mean.

$$
    \hat{y}(x) = \operatorname*{arg\,max}_{c \in \{1,\ldots, C \}} \frac{1}{C} \sum_{i=1}^{n} p_{i}^{(c)}(x)
$$

We can add weights to make more significant certain estimators at inference, but this feature is only available for "soft voting" strategy.

### Regression
Works as the same way as bagging regressors, just averages individual estimator outputs.

### Implementation
Scikit has the `VotingClassifier` and `VotingRegressor` classes. Both have the following hyperparameters:
- **`estimators`:** The estimators list.
- **`voting`:** Voting strategy for classification tasks, only avaliable at `VotingClassifier`.
- **`weights`:** Sequence of weights to weight the occurrences of predicted class labels, probabilities or values (depends on the task).

In [2]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.neighbors import RadiusNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

ESTIMATORS = [("RNN", RadiusNeighborsRegressor(radius=199, metric="euclidean", weights="distance", n_jobs=-1)),
             ("linear", LinearRegression())]

# Model definition
REGRESSION_MODEL = VotingRegressor(estimators=ESTIMATORS)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

Mean squared error: 3752118338.703068
Mean absolute error: 19430.052925


/usr/local/lib/python3.13/dist-packages/numpy/_core/numeric.py:457: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


## Stacking
Finally we got the stacked generalization approach, this is a method for combining estimators to reduce their biases. Individual estimators are trained using the entire dataset and then those are used as input to a final estimator to compute the final prediction and the final estimator is trained using cross validation. In practice this predictor predicts as good as the best predictor of the base layer and sometimes outperforms it by combining the strengths of these predictors. However this approach is computationally expensive as we need to train many estimators.

### Implementation
Scikit provides the `StackingClassifier` and `StackingRegressor` classes with the following hyperparameters:
- **`estimators`:** List of base estimators.
- **`final_estimator`:** The estimator which will be used to combine the base estimators.
- **`cv`:** Determines the cross-validation splitting strategy for training the final estimator.
- **`stack_method`:** Method that will be called to obtain the base estimator's outcome, only available at `StackingClassifier` class.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import StackingClassifier

FINAL_CLASSIFIER = LinearSVC(random_state=123, C=0.5)
CLASSIFIERS = [("tree", DecisionTreeClassifier(random_state=123, max_depth=5, criterion="gini")),
              ("MLP", MLPClassifier(random_state=123, activation="tanh", solver="adam"))]

# Model definition
CLASSIFICATION_MODEL = StackingClassifier(estimators=CLASSIFIERS, n_jobs=-1, final_estimator=FINAL_CLASSIFIER)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py:1380: RuntimeWarning: Number of classes in training fold (210) does not match total number of classes (225). Results may not be appropriate for your use case. To fix this, use a cross-validation technique resulting in properly stratified folds
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/m

Accuracy: 8.928571 %
Weighted F1-Score: 5.986955 %
